# STAR Project – Class Type Effects Over Time

**Goal:** For each grade (K to 8), estimate the effect of class type on math scores, with 95% confidence intervals.
Does the effect fade to zero after the experiment ends (after grade 3)?

**Three binary comparisons:**
1. Small vs Regular
2. Small vs Regular + Aide
3. Regular + Aide vs Regular

## 1. Load data and set up

Each student gets their **first** class type (the class type they were assigned when they entered the study).
This follows the original random assignment, even if the student later switched.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

df = pd.read_spss("../data/raw/STAR_Students.sav")
df = df.copy()

grades = ["gk", "g1", "g2", "g3", "g4", "g5", "g6", "g7", "g8"]
k3 = ["gk", "g1", "g2", "g3"]

classtype_cols = [f"{g}classtype" for g in k3]
df["first_classtype"] = df[classtype_cols].astype(object).bfill(axis=1).iloc[:, 0]
df["first_classtype"].value_counts()

## 2. Compute the effects

**Why standardize?** K–3 and 4–8 used different tests, so raw points can't be compared across grades.
In each grade we turn scores into z-scores (mean 0, SD 1). The effect is then in **standard deviations**.

For each grade and each comparison:
- **effect** = mean z of group A − mean z of group B
- **95% CI** = effect ± 1.96 × standard error
- **p-value** from a two-sided test (we keep it for the multiple-comparison correction later)

Note: this simple version does not control for school yet.

In [ ]:
comparisons = [
    ("SMALL CLASS", "REGULAR CLASS", "Small vs Regular"),
    ("SMALL CLASS", "REGULAR + AIDE CLASS", "Small vs Regular+Aide"),
    ("REGULAR + AIDE CLASS", "REGULAR CLASS", "Regular+Aide vs Regular"),
]

rows = []
for g in grades:
    score = df[f"{g}tmathss"]
    z = (score - score.mean()) / score.std()   # standardize within grade

    for a, b, label in comparisons:
        za = z[df["first_classtype"] == a].dropna()
        zb = z[df["first_classtype"] == b].dropna()

        effect = za.mean() - zb.mean()
        se = np.sqrt(za.var() / len(za) + zb.var() / len(zb))
        p = 2 * stats.norm.sf(abs(effect / se))

        rows.append({
            "grade": g, "comparison": label,
            "effect": effect, "ci_low": effect - 1.96 * se, "ci_high": effect + 1.96 * se,
            "p_value": p, "n_a": len(za), "n_b": len(zb),
        })

results = pd.DataFrame(rows)
results.round(3)

## 3. The graph

Each line is one comparison. Error bars are 95% confidence intervals.
The shaded area is the experiment years (K–3). The dashed line is zero (no effect).
If a confidence interval crosses zero, the effect in that grade is not significant.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(grades))
offsets = [-0.15, 0, 0.15]   # shift lines a bit so the error bars don't overlap

for (a, b, label), off in zip(comparisons, offsets):
    r = results[results["comparison"] == label]
    ax.errorbar(x + off, r["effect"],
                yerr=[r["effect"] - r["ci_low"], r["ci_high"] - r["effect"]],
                marker="o", capsize=3, label=label)

ax.axhline(0, color="gray", linestyle="--")
ax.axvspan(-0.5, 3.5, color="lightblue", alpha=0.3, label="Experiment years (K–3)")
ax.set_xticks(x)
ax.set_xticklabels(["K", "1", "2", "3", "4", "5", "6", "7", "8"])
ax.set_xlabel("Grade")
ax.set_ylabel("Effect on math score (SD)")
ax.set_title("Class type effects on math, by grade (95% CI)")
ax.legend()
plt.savefig("../figures/class_type_effects.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Quick summary table

Effect per grade for each comparison (in SD). `*` = 95% CI does not include zero.

In [ ]:
results["label"] = results["effect"].round(2).astype(str) + np.where(results["p_value"] < 0.05, "*", "")
results.pivot(index="grade", columns="comparison", values="label").loc[grades]

## Notes

_What do we see? Where does each effect cross zero?_

**Next step:** correct the p-values for multiple comparisons (3 comparisons × 9 grades = 27 tests).